# Specific conductance: implausible values and unit errors

**Why:** in the eligibility sensitivity review (2026-11-08, `mixed_model_results.ipynb` section 7b), one added dam, **MT-013**, made the before/after specific-conductance range 4× wider: its median went from 387 µS/cm before removal to **0.503 µS/cm** after. No river reads 0.5 µS/cm (that's close to distilled water); 0.503 looks like 503 µS/cm written in mS/cm. **NC-059** (286 → 43.5 µS/cm, 6 days after) is the next most extreme.

**What this notebook does (read-only; it changes nothing):**
1. MT-013's and NC-059's conductance records as reported: raw value, raw unit, organization, method, activity type, status. Is the unit wrong, and from when?
2. The whole cleaned table (`wq_measurement`): how many conductance values are implausibly low, and which stations / organizations / units they come from.
3. Stations whose conductance level jumps by ≥ 100× between years (the signature of a unit switch).
4. Which of those reach the analysis tables: main (3 / 2) and the sensitivity variants.

The result decides whether a cleaning rule is needed (applied to all data, main analysis included) and what it should be. Thresholds here are for screening, not rules.

In [ ]:
from pathlib import Path
import os
import sys

import duckdb
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from etl.common.config import load_config
from etl.common.warehouse import analysis_path, warehouse_dir

DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
CASE_STUDY = "all_removals"
WAREHOUSE = warehouse_dir(load_config(CASE_STUDY, project_root=DATA_ROOT))
CHECK_DAMS = ["MT-013", "NC-059"]
LOW_US_CM = 5.0       # screening: below this is implausible for river water
JUMP_FACTOR = 100.0   # screening: year-to-year change in a station's median level

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

con = duckdb.connect()
con.execute(f"CREATE VIEW wq AS SELECT * FROM read_parquet('{WAREHOUSE / 'wq_measurement.parquet'}') "
            "WHERE parameter_std = 'specific_conductance'")
TABLES = {"main (3 / 2)": analysis_path(WAREHOUSE)}
for v in ("min1yr", "min1yr_2mo"):
    if analysis_path(WAREHOUSE, v).exists():
        TABLES[v] = analysis_path(WAREHOUSE, v)
for name, path in TABLES.items():
    con.execute(f"CREATE VIEW \"{name}\" AS SELECT * FROM read_parquet('{path}') "
                "WHERE parameter_std = 'specific_conductance'")
print(f"Warehouse: {WAREHOUSE}")
print(con.sql("SELECT count(*) AS conductance_results, count(DISTINCT station_id) AS stations FROM wq").df())

## 1. MT-013 and NC-059 as reported

Per dam, station, side and period: how many results, the median cleaned value, and the raw units, organizations and methods behind them. Then every result after removal. **What to look for:** a different organization, method or raw unit after removal, or raw values that are ~1000× smaller than before (mS/cm reported as µS/cm).

In [ ]:
src = "min1yr" if "min1yr" in TABLES else "main (3 / 2)"
dams = ", ".join(f"'{d}'" for d in CHECK_DAMS)
rows = con.sql(f"""
    SELECT a.AR_ID, a.station_id, a.side, a.period, a.sample_date, a.network_distance_km,
           w.result_value_raw, w.result_unit_raw, w.result_value_std, w.organization_id, w.provider,
           w.method_id, w.activity_type, w.result_status, w.value_type, w.qualifier_code, w.parameter
    FROM "{src}" a JOIN wq w USING (measurement_id)
    WHERE a.AR_ID IN ({dams})
    ORDER BY a.AR_ID, a.station_id, a.sample_date
""").df()
print(f"From {src}: {len(rows)} conductance results at {rows['station_id'].nunique()} stations")
summary = (rows.groupby(["AR_ID", "station_id", "side", "period"])
           .agg(results=("result_value_std", "size"), median_us_cm=("result_value_std", "median"),
                min_us_cm=("result_value_std", "min"), max_us_cm=("result_value_std", "max"),
                first=("sample_date", "min"), last=("sample_date", "max"),
                raw_units=("result_unit_raw", lambda s: sorted(s.dropna().unique())),
                organizations=("organization_id", lambda s: sorted(s.dropna().unique())),
                methods=("method_id", lambda s: sorted(s.dropna().unique())),
                activity_types=("activity_type", lambda s: sorted(s.dropna().unique())))
           .reset_index())
display(summary)
print("Every result after removal:")
display(rows[rows["period"] == "post"])

## 2. Implausibly low conductance in the whole cleaned table

Results below the screening value (5 µS/cm), by organization, raw unit and method. A cluster at one organization or one raw unit points to a reporting or unit error rather than real water.

In [ ]:
low = con.sql(f"""
    SELECT organization_id, provider, result_unit_raw, method_id,
           count(*) AS results, count(DISTINCT station_id) AS stations,
           median(result_value_std) AS median_us_cm, min(sample_date) AS first, max(sample_date) AS last
    FROM wq WHERE result_value_std < {LOW_US_CM} AND NOT is_censored
    GROUP BY ALL ORDER BY results DESC
""").df()
total = con.sql(f"SELECT count(*) FROM wq WHERE result_value_std < {LOW_US_CM} AND NOT is_censored").fetchone()[0]
print(f"Conductance results below {LOW_US_CM} µS/cm (not censored): {total:,}")
display(low.head(40))

## 3. Stations whose level jumps between years

Per station and year, the median conductance; flagged where it changes by ≥ 100× from the station's previous year with data. Real changes in a river's conductance are rarely that large; a unit switch (µS/cm ↔ mS/cm) is exactly 1000×.

In [ ]:
jumps = con.sql(f"""
    WITH yearly AS (
        SELECT station_id, year(sample_date) AS year, median(result_value_std) AS median_us_cm,
               count(*) AS results, string_agg(DISTINCT result_unit_raw, ', ') AS raw_units,
               string_agg(DISTINCT organization_id, ', ') AS organizations
        FROM wq WHERE NOT is_censored AND result_value_std > 0
        GROUP BY ALL
    ), stepped AS (
        SELECT *, lag(median_us_cm) OVER (PARTITION BY station_id ORDER BY year) AS prev_median,
                  lag(year) OVER (PARTITION BY station_id ORDER BY year) AS prev_year
        FROM yearly
    )
    SELECT station_id, prev_year, year, prev_median, median_us_cm, median_us_cm / prev_median AS ratio,
           results, raw_units, organizations
    FROM stepped
    WHERE prev_median IS NOT NULL
      AND (median_us_cm / prev_median >= {JUMP_FACTOR} OR prev_median / median_us_cm >= {JUMP_FACTOR})
    ORDER BY station_id, year
""").df()
print(f"Station-years with a ≥{JUMP_FACTOR:.0f}× step: {len(jumps)} at {jumps['station_id'].nunique()} stations")
display(jumps)

## 4. Do they reach the analysis tables?

For each analysis table: the dams and stations with low values or a jump, and how many conductance rows that is. **If the main (3 / 2) table has none, fixing this changes only the sensitivity results.**

In [ ]:
bad_stations = set(jumps["station_id"])
bad_low = set(con.sql(f"SELECT DISTINCT station_id FROM wq WHERE result_value_std < {LOW_US_CM} AND NOT is_censored").df()["station_id"])
reach = []
for name in TABLES:
    t = con.sql(f'SELECT AR_ID, station_id, period, result_value_std FROM "{name}"').df()
    for what, ids in (("low value station", bad_low), ("jump station", bad_stations)):
        hit = t[t["station_id"].isin(ids)]
        reach.append({"table": name, "screen": what, "dams": hit["AR_ID"].nunique(),
                      "stations": hit["station_id"].nunique(), "rows": len(hit),
                      "rows_below_screen": int((hit["result_value_std"] < LOW_US_CM).sum()),
                      "dam_ids": ", ".join(sorted(hit["AR_ID"].unique())[:15])})
display(pd.DataFrame(reach))

**Next step (decided after reading this, not here):** if the problem is confined to a recognizable pattern (an organization, a raw unit, a date), the fix belongs in `etl/clean_wq_results.py` / `etl/common/wq_rules.py` as a documented rule applied to every result: either convert (if the unit error is certain) or drop and count. Then rebuild the analysis tables and refit, and record the change in `docs/eligibility_sensitivity_prespec.md`. No dam is excluded by hand.